# Silver MOPEP com GPT-5

Fluxo: carregar um split, montar o prompt, testar uma empresa e repetir a mesma chamada sequencialmente para uma amostra ou para todo o split.

## 1. Imports e configuração

In [ ]:
import json
import os
import sys
from pathlib import Path
from time import perf_counter

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "analytics" / "src").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analytics.src.evaluate_mopep_tags import (
    ALLOWED_TAGS,
    TAG_DEFINITIONS,
    compare_tags,
    csv_ready,
    load_dataset,
    normalize_tags,
    summarize,
)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

DATASETS_DIR = Path(os.getenv("MOPEP_BMC_DATA_DIR", "/tmp/mopep-bmc-tags"))
SPLIT = "calibration"       # train, calibration ou test
MODEL = "gpt-5-nano"       # opção mais barata da família GPT-5
LIMIT = None                    # use None para processar todo o split
RANDOM_STATE = 42
MAX_OUTPUT_TOKENS = 256
OUTPUT_PATH = REPO_ROOT / f"analytics/results/openai-silver/{SPLIT}-{MODEL}.csv"
OPENAI_API_KEY = ""
# if not os.getenv("OPENAI_API_KEY"):
#     raise RuntimeError("Defina OPENAI_API_KEY antes de executar o notebook.")

client = OpenAI(api_key=OPENAI_API_KEY)

## 2. Carregar os dados

`expected_tags` fica local e é usado somente para avaliar a resposta; ele não entra no prompt.

In [ ]:
dataset_path = DATASETS_DIR / f"{SPLIT}.csv"
dataset_df = load_dataset(dataset_path)
print(f"{len(dataset_df)} exemplos carregados de {dataset_path}")
dataset_df[["example_id", "business_model", "expected_tags"]].head()

## 3. Construir o prompt

In [ ]:
TAG_LIST = "\n".join(
    f"- {name}: {description}" for name, description in TAG_DEFINITIONS
)

PROMPT_TAGS = """
Seu objetivo é analisar o modelo de negócios de uma empresa e classificá-la usando somente as tags da lista abaixo:

- empresa estabelecida: As evidências indicam que a empresa já é estabelecida, com mais de 5 anos de existência.
- estabelecimento físico: As evidências apontam que a empresa possui um estabelecimento físico ou escritório onde a atividade principal é desempenhada.
- b2b: A empresa vende produtos ou serviços para outras empresas.
- vendas remotas: As negociações e vendas são realizadas remotamente, por telefone, e-mail, WhatsApp, videoconferência ou outros canais digitais, sem necessidade de visita presencial.
- serviço: A empresa oferece soluções intangíveis, como consultoria, manutenção, suporte técnico ou treinamentos, entregando valor por meio de conhecimento e experiência.
- atuação nacional: A empresa realiza operações ou atende clientes em todo o território brasileiro, com presença ou cobertura em múltiplas regiões ou estados.
- tecnologia: A empresa se apresenta como inovadora no uso de tecnologia em sua atuação, produtos ou serviços.
- produto: A empresa comercializa bens tangíveis ou digitais, como equipamentos, roupas, alimentos, softwares, e-books ou outros produtos.
- whatsapp: O site possui indícios de que a empresa realiza atendimento ou vendas por WhatsApp.
- b2c: A empresa vende produtos ou serviços diretamente para o consumidor final.
- instagram: O site possui link apontando para o perfil da empresa no Instagram.
- compra planejada: Os clientes costumam pesquisar, comparar alternativas ou planejar antes de comprar os produtos ou contratar os serviços.
- blog: O site possui um blog ou área dedicada à publicação de artigos, notícias ou conteúdos informativos.
- público: empresas médias e grandes: A empresa B2B ou B2B2C se posiciona com ênfase no atendimento a empresas médias e grandes, com maior faturamento, estrutura ou número de funcionários.
- linkedin: O site possui link apontando para o perfil da empresa no LinkedIn.
- atuação regional: A empresa atua predominantemente em uma região geográfica específica, como um estado, grupo de municípios ou área delimitada.
- facebook: O site possui link apontando para o perfil da empresa no Facebook.
- atuação internacional: A empresa possui operações, clientes ou negócios em mais de um país, atuando em mercados estrangeiros ou globalmente.
- público: micro e pequenas empresas: A empresa B2B ou B2B2C se posiciona com ênfase no atendimento a MEIs, microempresas ou pequenas empresas.
- segmento: tecnologia da informação: A empresa tem como atividade principal a oferta de soluções de tecnologia da informação, como desenvolvimento de software, infraestrutura, redes, segurança da informação ou suporte técnico.

O modelo de negócios da empresa é:

{business_model}

Retorne somente uma lista Python com as tags aplicáveis.

Exemplo de resposta:

['empresa estabelecida', 'b2b', 'serviço', 'vendas remotas']

Procedimento:
- Avalie individualmente cada uma das 20 tags antes de responder.
- Considere todas as informações presentes no texto, mesmo quando algumas seções do Business Model Canvas estiverem vazias.
- Inclua uma tag quando houver evidência direta ou uma inferência razoável sustentada pelo texto.
- Não deixe de classificar a empresa apenas porque o documento está incompleto.
- Use [] somente depois de avaliar todas as tags e concluir que nenhuma delas pode ser sustentada.
- Retorne somente a lista final, sem mostrar a análise.

Regras:
- Use somente as tags apresentadas na lista.
- Copie exatamente o nome das tags, incluindo acentos.
- Não crie tags novas.
- Não inclua explicações ou qualquer texto fora da lista.
# - Não inclua uma tag quando não houver evidência suficiente.
# - Caso nenhuma tag seja aplicável, retorne [].
""".strip()

def build_prompt(business_model):
    return PROMPT_TAGS.format(
        business_model=business_model,
    )

In [ ]:
example = dataset_df.iloc[0]
prompt = build_prompt(example["business_model"])
print(prompt)

## 4. Fazer uma consulta

A saída estruturada obriga a API a retornar um objeto JSON no formato `{'tags': [...]}`.

In [ ]:
OUTPUT_FORMAT = {
    "type": "json_schema",
    "name": "mopep_tags",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "tags": {
                "type": "array",
                "items": {"type": "string", "enum": ALLOWED_TAGS},
            }
        },
        "required": ["tags"],
        "additionalProperties": False,
    },
}

In [ ]:
def predict_one(example):
    started_at = perf_counter()
    response = client.responses.create(
        model=MODEL,
        input=build_prompt(str(example["business_model"])),
        reasoning={"effort": "minimal"},
        text={"format": OUTPUT_FORMAT},
        max_output_tokens=MAX_OUTPUT_TOKENS,
        store=False,
    )
    latency_ms = (perf_counter() - started_at) * 1000
    generated_tags = normalize_tags(json.loads(response.output_text)["tags"])

    result = {
        "example_id": example["example_id"],
        "model": MODEL,
        "expected_tags": example["expected_tags"],
        "generated_tags": generated_tags,
        "raw_response": response.output_text,
        "request_ok": True,
        "parse_ok": True,
        "latency_ms": latency_ms,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }
    result.update(compare_tags(result["expected_tags"], generated_tags))
    return result

In [ ]:
single_prediction = predict_one(example)
single_prediction

## 5. Repetir para uma amostra ou para todo o split

As requisições são sequenciais. Comece com `LIMIT = 10`; depois use `None` para processar tudo.

In [ ]:
def predict_table(dataset_path, output_path):
    dataset_path = Path(dataset_path)
    output_path = Path(output_path)

    # 1. Carrega a tabela
    dataset_df = load_dataset(dataset_path)

    # 2. Prevê cada linha sequencialmente
    predictions = []

    for _, example in tqdm(
        dataset_df.iterrows(),
        total=len(dataset_df),
        desc=dataset_path.stem,
    ):
        predictions.append(predict_one(example))

    predictions_df = pd.DataFrame(predictions)

    # 3. Salva a tabela gerada
    output_path.parent.mkdir(parents=True, exist_ok=True)

    csv_ready(predictions_df).to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )

    print(
        f"{len(predictions_df)} predições salvas em "
        f"{output_path}"
    )

    return predictions_df

In [ ]:
OUTPUT_ROOT = REPO_ROOT / "analytics/results/openai-silver"

## Calibração

In [ ]:
calibration_predictions = predict_table(
    dataset_path=DATASETS_DIR / "calibration.csv",
    output_path=OUTPUT_ROOT / "calibration.csv",
)

calibration_predictions.head()

In [ ]:
test_predictions = predict_table(
    dataset_path=DATASETS_DIR / "test.csv",
    output_path=OUTPUT_ROOT / "test.csv",
)

test_predictions.head()

## Treino

## 6. Resumir e salvar

In [ ]:
performance_df = summarize(predictions_df)
performance_df["input_tokens"] = predictions_df["input_tokens"].sum()
performance_df["output_tokens"] = predictions_df["output_tokens"].sum()
performance_df

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
csv_ready(predictions_df).to_csv(OUTPUT_PATH, index=False)
print(f"Predições salvas em {OUTPUT_PATH}")